# 02 — Weak learners and the additive model

*Chapter 07 — AdaBoost · Notebook 2 of 5*

Notebook 1 built the engine: reweight the points, fit a weak learner, and trust it with a vote weight
α = ln((1 − ε)/ε). We took two things on faith. First, *how* do the stumps combine into a single
prediction? Second, *where* did that formula for α come from? This notebook answers both — and they
turn out to be one answer. AdaBoost builds a **weighted additive model**, and α is the step that
minimises a **loss**. This is the chapter's densest notebook; we build the idea one picture at a time.

**Prerequisites.**
- **Notebook 1:** the reweighting loop, the weighted error ε, the learner weight α, and the by-hand /
  `AdaBoostClassifier` parity.
- **Chapter 04:** the decision stump (a depth-1 tree), our weak learner.
- **Chapter 03, NB 3:** a **loss** is a single number we drive down; **log-loss** punished a confident
  wrong prediction without bound, where squared error flattens out. We lean on that idea here.
- **Module 00:** the decision boundary and accuracy.

**What you'll be able to do.**
- Write AdaBoost's prediction as a **weighted additive vote** `F(x) = sign(Σ αₜ hₜ(x))`.
- Watch the decision boundary **sharpen** as rounds are added.
- **Derive** α as the value that minimises the **exponential loss** — not a formula handed down.
- Reconcile scikit-learn's SAMME α with the classic ½ form, and state the multiclass rule.

## Where we are — from a loop to a model

Notebook 1 gave us a *procedure*: reweight, fit a stump, repeat. But a procedure is not yet a model —
we never wrote down the function that turns a new point into a prediction. AdaBoost's model is the
**weighted additive vote**: add up every weak learner's call, each scaled by its α, and read off the
sign.

And α itself we used on faith. Here we earn it. We will see that α = ln((1 − ε)/ε) is not arbitrary: it
is the step that drives down a specific **loss function**, the same way gradient descent drove down
log-loss in chapter 03. Two open questions from NB 1, one notebook, one underlying idea.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_classification, make_moons
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

from ml_course import viz
from ml_course.colors import COLORS

viz.use_course_style()
SEED = 0

# The same two crescents as notebook 1.
X, y = make_moons(n_samples=400, noise=0.20, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
print(f"train: {X_train.shape[0]} points    test: {X_test.shape[0]} points")

## The weighted additive vote

After T rounds we hold T weak learners h₁, …, h_T and their weights α₁, …, α_T. AdaBoost combines them
into a single score and takes its sign:

$$F(x) = \sum_{t=1}^{T} \alpha_t\, h_t(x), \qquad \hat{y}(x) = \operatorname{sign} F(x).$$

Here is the reveal. The **same** α that decided how hard to reweight the data in NB 1 is also each
learner's **vote weight**. A learner that scored a low error ε earns a large α — it reweighted the data
sharply *and* it speaks loudly in the final vote. Chapter 06's bagging gave every tree an **equal**
vote; boosting gives a **weighted** one, and the weight is α. (We *use* this vote now; below, we derive
where its α comes from.)

In [ ]:
# The boundary of the weighted additive vote, as rounds accumulate.
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for T, ax in zip((1, 10, 50), axes, strict=False):
    model = AdaBoostClassifier(n_estimators=T, random_state=SEED).fit(X_train, y_train)
    viz.plot_decision_boundary(model, X_train, y_train, ax=ax)
    ax.set_title(f"T = {T} rounds   (test acc {model.score(X_test, y_test):.3f})")
plt.show()

**Read the figure.** At T = 1 the model is a single stump — one straight cut, the 0.867 we saw in
NB 1. By T = 10 the weighted sum of ten axis-aligned cuts has become a coarse **staircase** that already
scores 0.942; by T = 50 the staircase is fine enough to follow the two crescents closely. No individual
stump ever curved — each is still one straight, axis-aligned line. The **sum** grew expressive: a
boundary that tracks a curve, built from many simple straight pieces — a staircase that refines with
every round.

## What counts as a weak learner

A **weak learner** needs only to beat chance: a weighted error ε < ½, which makes α = ln((1 − ε)/ε) > 0.
That is a low bar — a single stump clears it — and it is all boosting requires. Why does adding such
modest pieces work? Each term αₜ hₜ(x) is a small correction to the running score F. Stacking many of
them builds a flexible function out of simple parts, the way a curve can be approximated by adding more
and more terms. The craft is in *choosing* each term — which is where the loss comes in.

## Where does α come from?

In NB 1 we used α = ln((1 − ε)/ε) without justification. Time to earn it. Recall chapter 03: a **loss**
is a single number measuring how wrong a model is, and training means driving it down. Log-loss had a
telling shape — it punished a *confident* wrong prediction without bound, while a cautious one paid
little. AdaBoost is built on a close cousin with the same spirit: the **exponential loss**. Let us look
at it before we use it.

In [ ]:
margin = np.linspace(-2, 2, 400)
exp_loss = np.exp(-margin)
zero_one = (margin < 0).astype(float)  # 0/1 loss: 1 when wrong (margin < 0), else 0

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(margin, exp_loss, color=COLORS["model"], linewidth=2.4,
        label="exponential loss  exp(-margin)")
ax.plot(margin, zero_one, color=COLORS["error"], linewidth=2.0, linestyle="--",
        label="0/1 loss (the step)")
ax.axvline(0, color=COLORS["muted"], linewidth=1.0)
ax.set_xlabel("margin  =  y · F(x)")
ax.set_ylabel("loss")
ax.set_ylim(0, 5)
ax.legend(loc="upper right")
plt.show()

**Read the figure.** The horizontal axis is the **margin**, y · F(x): positive when the model is
right, negative when wrong, larger in size the more confident the call. The dashed step is the 0/1 loss
— 1 for any mistake, 0 otherwise — the thing we ultimately care about, but flat and unusable for
optimisation. The exponential loss exp(−margin) is its smooth stand-in: nearly 0 for confident-correct
points, and it **blows up exponentially** as a point goes confidently wrong. That smoothness is what
lets us optimise; the exponential blow-up on wrong points is also the seed of a weakness — a
**mislabeled** point looks confidently wrong and is punished enormously, dragging the model toward it.
We meet that in notebooks 3 and 5.

## Forward stagewise: one term at a time

We do not optimise all T terms at once — we add them greedily. Freeze the model built so far, F_{t−1},
and add one new term: F_t = F_{t−1} + α·h. Choose the weak learner h and its weight α to most reduce the
total exponential loss over the training set,

$$L = \sum_i \exp\!\big(-y_i\,[F_{t-1}(x_i) + \alpha\, h(x_i)]\big).$$

This is **forward stagewise additive modelling** (Friedman, Hastie & Tibshirani, 2000). Fix the weak
learner h with weighted error ε; what single α minimises L? Splitting the sum into the points h gets
right and the points it gets wrong, the weighted loss collapses to

$$L(\alpha) = (1 - \varepsilon)\,e^{-\alpha} + \varepsilon\,e^{\alpha}.$$

Let us find its bottom — first numerically, then in closed form.

In [ ]:
# Round 1, uniform weights: read off the weighted error, then find the alpha that
# minimises L(alpha) = (1 - eps) e^{-alpha} + eps e^{alpha}.
y_pm = np.where(y_train == 1, 1, -1)
w = np.full(len(y_pm), 1.0 / len(y_pm))
stump = DecisionTreeClassifier(max_depth=1, random_state=SEED)
stump.fit(X_train, y_train, sample_weight=w)
pred = np.where(stump.predict(X_train) == 1, 1, -1)
eps = w[pred != y_pm].sum() / w.sum()

alphas = np.linspace(-0.5, 2.5, 6001)
margins = (y_pm * pred).astype(float)  # +1 on correct points, -1 on wrong ones
loss = np.array([np.sum(w * np.exp(-a * margins)) for a in alphas])
alpha_grid = alphas[np.argmin(loss)]
alpha_star = 0.5 * np.log((1 - eps) / eps)

print(f"weighted error eps = {eps:.4f}")
print(f"grid minimiser of L(alpha)        = {alpha_grid:.4f}")
print(f"closed form   1/2 ln((1-eps)/eps) = {alpha_star:.4f}")
print(f"SAMME alpha       ln((1-eps)/eps) = {2 * alpha_star:.4f}  (= 2 x the minimiser)")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(alphas, loss, color=COLORS["model"], linewidth=2.4)
ax.scatter([alpha_star], [loss[np.argmin(np.abs(alphas - alpha_star))]],
           color=COLORS["highlight"], s=90, zorder=5,
           label=f"minimiser  ½ln((1-ε)/ε) = {alpha_star:.2f}")
ax.axvline(2 * alpha_star, color=COLORS["muted"], linestyle="--", linewidth=1.3,
           label=f"SAMME α = {2 * alpha_star:.2f}")
ax.set_xlabel("α  (this round's vote weight)")
ax.set_ylabel("exponential loss  L(α)")
ax.legend(loc="upper center")
plt.show()

**Read the figure.** L(α) is a convex bowl. Setting its derivative to zero,
−(1 − ε)e^(−α) + ε e^(α) = 0, gives e^(2α) = (1 − ε)/ε, so the minimiser is

$$\alpha^* = \tfrac{1}{2}\,\ln\frac{1-\varepsilon}{\varepsilon}.$$

For ε = 0.157 that minimiser is **0.84** — and the grid search lands on the same value. This is where α
comes from: the step that minimises the exponential loss, nothing arbitrary about it. Its shape even
carries NB 1's behaviour — as ε → 0 the bottom slides to α → ∞ (trust a near-perfect learner), and at
ε = ½ the bowl's bottom sits at α = 0 (a coin flip earns no vote).

## Two conventions, one classifier

The minimiser we derived is the **vote weight** β = ½ ln((1 − ε)/ε) = 0.84. Yet notebook 1 — and
scikit-learn's SAMME — used ln((1 − ε)/ε) = 1.68, exactly **twice** as large. Two different α's: do
they give different models? They give the **same** one, and the reason is worth seeing. α appears in
two places, so check both.

- **The vote** is sign(Σ α h). Scaling every α by the same positive factor cannot flip a sign, so the
  β-vote and the 2β-vote classify identically — *provided the stumps are the same*.
- **The reweighting** is where it could go wrong, and where the real argument lives. The exp-loss
  derivation reweights each point by the **margin form** exp(−β yᵢ hᵢ): a correct point (yᵢhᵢ = +1) is
  multiplied by exp(−β), a wrong one (yᵢhᵢ = −1) by exp(+β). Pull out the common exp(−β):

$$w_i \leftarrow w_i\, e^{-\beta y_i h_i} = w_i\, e^{-\beta}\, e^{\,2\beta\,\mathbb{1}[\text{miss}]}.$$

The shared exp(−β) is identical for every point, so it **vanishes when we renormalise**. What survives
multiplies the *misclassified* weights by exp(2β) = (1 − ε)/ε — exactly the SAMME update from notebook
1 (multiply misses by exp(α), α = ln((1 − ε)/ε) = 2β).

So the two conventions **reweight identically**, build the **same sequence of stumps**, and their votes
differ only by a global factor of 2 that the sign erases: the **same classifier**. (Careful — this
rests on the *margin* form. Reweighting misses by exp(β) instead of exp(2β), the naive "halve
everything", would *not* match: α sits in an exponent, so only the common term cancels, not the whole
factor of 2.) The dashed line marks where SAMME's α sits on the bowl.

In [ ]:
# Multiclass SAMME adds one term, ln(K - 1). Check it against sklearn on a 3-class set.
Xm, ym = make_classification(
    n_samples=600, n_features=6, n_informative=4, n_redundant=0,
    n_classes=3, n_clusters_per_class=1, random_state=SEED,
)
Xm_tr, _, ym_tr, _ = train_test_split(
    Xm, ym, test_size=0.30, random_state=SEED, stratify=ym
)
K = 3
wm = np.full(len(ym_tr), 1.0 / len(ym_tr))
hm = DecisionTreeClassifier(max_depth=1, random_state=SEED)
hm.fit(Xm_tr, ym_tr, sample_weight=wm)
eps_m = wm[hm.predict(Xm_tr) != ym_tr].sum() / wm.sum()
alpha_byhand = np.log((1 - eps_m) / eps_m) + np.log(K - 1)
ada_m = AdaBoostClassifier(n_estimators=10, random_state=SEED).fit(Xm_tr, ym_tr)

print(f"K = {K} classes,  round-1 weighted error eps = {eps_m:.4f}")
print(f"by-hand  ln((1-eps)/eps) + ln(K-1) = {alpha_byhand:.4f}")
print(f"sklearn  estimator_weights_[0]     = {ada_m.estimator_weights_[0]:.4f}")

**Read the result.** scikit-learn ships only SAMME, and SAMME extends to K classes with a single
extra term: α = ln((1 − ε)/ε) + **ln(K − 1)**. For K = 2 that term is ln(1) = 0, recovering the binary
form derived above. For K = 3 it adds ln 2 ≈ 0.69, and the by-hand value matches `estimator_weights_` to
the last digit. The "better than chance" bar also rises with K: a multiclass weak learner must beat
ε < 1 − 1/K (random guessing among K classes), otherwise α turns negative.

## What we built, and what is still open

You turned NB 1's procedure into a model — the weighted additive vote F = sign(Σ αₜ hₜ) — and
**derived** α as the exponential-loss minimiser. Four honest notes.

- **Exponential loss is a surrogate.** We do not optimise the 0/1 error directly (it is flat and
  unusable); we optimise a smooth stand-in, chosen because its per-round minimiser is clean. A different
  surrogate gives a different boosting algorithm — chapter 08 swaps it out.
- **That surrogate is why noise hurts.** The exponential penalty on confidently-wrong points means a
  mislabeled example is punished without bound and hoards the model's attention — AdaBoost's known weak
  spot, measured in notebooks 3 and 5.
- **Greedy, not global.** Forward stagewise adds one term at a time and never refits earlier stumps, so
  the model is a greedy approximation, not a global optimum.
- **It fits the training set perfectly.** This additive model drove training error to 0 by about 114
  rounds. Expressive — but whether that *generalises*, and how the learning rate controls it, is
  notebook 3.

## Your turn

1. **Read the staircase (easy).** In Figure A, the T = 1 boundary is a single straight line. By T = 10,
   has the boundary become a finer, multi-step staircase that follows the crescents more closely — and
   does its test accuracy already beat the single stump's 0.867? (Read the accuracy off the panel
   titles.)
2. **Two conventions (medium).** Compute the minimiser ½ ln((1 − ε)/ε) and the SAMME value
   ln((1 − ε)/ε) by hand for ε = 0.1 and ε = 0.3. Confirm SAMME = 2 × the minimiser in each case, and
   that both grow as ε shrinks.
3. **The doubling puzzle (harder).** Show that plugging the SAMME step (twice the minimiser) into
   L(α) = (1 − ε)e^(−α) + ε e^(α) gives the *same* exponential loss as taking no step at all,
   L(2β) = L(0) = 1. Yet the SAMME model classifies identically to the β model. Resolve the apparent
   contradiction. (Hint: classification depends on the *sign* of the vote, not the loss value; and from
   the section above, the two conventions reweight identically — so they build the same stumps, and
   SAMME merely votes with double the coefficient, which the sign ignores.)

## What you built

- You wrote AdaBoost's prediction as a **weighted additive vote**, `F(x) = sign(Σ αₜ hₜ(x))`, and saw
  the **same α** play two roles — reweighting strength (NB 1) and vote weight (here).
- You watched the decision boundary **sharpen** from one straight cut (T = 1) to a fine axis-aligned
  staircase that follows the crescents (T = 50), built entirely from weighted stumps.
- You **derived** α = ½ ln((1 − ε)/ε) as the minimiser of the **exponential loss**, by forward stagewise
  additive modelling — confirmed by a grid search.
- You reconciled the classic ½ form with scikit-learn's **SAMME** (twice as large, same classifier) and
  stated the multiclass rule, + ln(K − 1), verified against `estimator_weights_`.

**Vocabulary you now own:** additive model · weighted vote · weak learner (ε < ½) · margin · exponential
loss · surrogate loss · forward stagewise additive modelling · SAMME vs the classic ½ · multiclass
+ ln(K − 1).

## References

- Freund, Y., & Schapire, R. E. (1997). *A decision-theoretic generalization of on-line learning and an
  application to boosting.* Journal of Computer and System Sciences 55(1), 119–139.
  DOI: [10.1006/jcss.1997.1504](https://doi.org/10.1006/jcss.1997.1504)
- Friedman, J., Hastie, T., & Tibshirani, R. (2000). *Additive logistic regression: a statistical view
  of boosting.* The Annals of Statistics 28(2), 337–407.
  DOI: [10.1214/aos/1016218223](https://doi.org/10.1214/aos/1016218223)
- Zhu, J., Zou, H., Rosset, S., & Hastie, T. (2009). *Multi-class AdaBoost.* Statistics and Its
  Interface 2(3), 349–360. DOI: [10.4310/SII.2009.v2.n3.a8](https://doi.org/10.4310/SII.2009.v2.n3.a8)
- Hastie, T., Tibshirani, R., & Friedman, J. (2009). *The Elements of Statistical Learning* (2nd ed.),
  §10.1–10.4 (boosting and the exponential loss).
  DOI: [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7)

*Previous: 01 — Boosting intuition: focus on the mistakes. Next: 03 — Learning rate, number of rounds,
and overfitting behaviour.*